# 03 · Targeted Relabelling Pass

The v1 model reached 71.2% test accuracy but only 24% recall on `right`.
Error analysis showed most `right`→`straight` confusions came from one video,
and visual inspection revealed **labeller bias**: frames had been labelled
by the driver's *intended* turn at a corner rather than what the line looked
like in the current frame. This notebook re-reviews every `left`/`right`
frame from the forward-direction videos (where this bias was concentrated)
and corrects the label to match visual content.

269 frames were corrected this way (197 `right`, 72 `left`), mostly
relabelled to `straight` — see Table I / Fig. 1 in the report.

Keys: `h`=hard_left, `k`=hard_right, `l`=left, `r`=right, `s`=straight,
`j`=skip, `q`=quit.

In [ ]:
import cv2
import pandas as pd
from matplotlib import pyplot as plt
from IPython.display import clear_output

# load existing labels
df = pd.read_csv("labels.xls")

# forward-direction videos where left/right labels are often
# skewed toward the driver's intended turn rather than the visible line
forward_videos = [
    "color_video", "color_video (1)", "color_video (2)",
    "color_video (3)", "color_video (5)", "color_video (6)", "color_video (7)",
]

df["video"] = df["image"].apply(lambda p: p.split("\\")[1])

suspects = df[
    (df["video"].isin(forward_videos)) &
    (df["label"] == "left")
].reset_index(drop=True)

print(f"Found {len(suspects)} forward left frames to review")
print("h=hard_left, k=hard_right, l=left, r=right, s=straight, j=skip, q=quit")

label_map = {
    "h": "hard_left",
    "k": "hard_right",
    "l": "left",
    "r": "right",
    "s": "straight",
    "j": None,
}

corrections = {}  # image_path -> new_label

for i, row in suspects.iterrows():
    img_path = row["image"]
    img = cv2.imread(img_path)

    if img is None:
        print(f"Failed to load: {img_path}")
        continue

    # crop top of image to match training preprocessing
    if img.shape[0] > 200:
        img = img[200:, :]

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(8, 6))
    plt.imshow(img_rgb)
    plt.title(f"{i+1}/{len(suspects)} | Current: {row['label']} | "
              f"h=hard_left  k=hard_right  l=left  r=right  s=straight j=skip  q=quit")
    plt.axis("off")
    plt.show()

    key = input("New label: ").strip().lower()
    clear_output(wait=True)

    if key == "q":
        print("Stopped early")
        break
    elif key in label_map and label_map[key] is not None:
        corrections[img_path] = label_map[key]
        print(f"{img_path} -> {label_map[key]}")
    else:
        print("Skipped")

print(f"\n{len(corrections)} labels changed")
for path, new_label in corrections.items():
    df.loc[df["image"] == path, "label"] = new_label

## Save corrected labels

In [ ]:
# save labels without the helper 'video' column
df.drop(columns=["video"]).to_csv("labels.xls", index=False)
print("Saved updated labels.xls")
print("\nNew distribution:")
print(df["label"].value_counts())